In [7]:
from baja.toy_problem import (
    load_cubs_traj_data,
    get_toy_setup_params,
    dynamics_fn,
    log_measurement,
)
import numpy as np
import matplotlib.pyplot as plt
import rerun as rr
from baja import ukf, fpf, GaussianState, ParticleState
import jax.numpy as jnp
import jax.random as jr

df = load_cubs_traj_data("0.1s")[:200]
# df = pd.read_parquet("../data/cub_sample_traj.parquet")
seq_len = len(df)

ts = df.index.to_numpy().astype(np.datetime64)
target_pos = df[["x", "y", "z"]].to_numpy()

sensor_type = "b"
Q, R, meas_fn, sensor_pos = get_toy_setup_params(sensor_type)
meas_dim = R.shape[0]

In [8]:
key = jr.key(111)
rng = np.random.default_rng(12345)


init_state = np.hstack([rng.normal(target_pos[0], size=(3,)), rng.uniform(-1, 1, (3,))])
state_ukf = GaussianState(mean=init_state, cov=Q)

filter_ukf = ukf.UnscentedKalmanFilter(
    f=dynamics_fn,
    h=meas_fn,
    Q=Q,
    R=R,
)

n_particles = 100
init_state = np.hstack(
    [
        rng.normal(target_pos[0], size=(n_particles, 3)),
        rng.uniform(-1, 1, (n_particles, 3)),
    ]
)
state_fpf = ParticleState(init_state, np.ones(n_particles) / n_particles, key)
filter_fpf = fpf.FPF(
    f=dynamics_fn,
    h=meas_fn,
    num_particles=n_particles,
    Q=Q,
    R=R,
    flow_steps=100,
)

In [ ]:
rr.init("traj_vis")
ukf_hist = []
ukf_hist.append(state_ukf)

fpf_hist = []
fpf_hist.append(state_fpf)

rr.log(
    "sensors/location",
    rr.Points3D(
        positions=sensor_pos,
        radii=[0.2] * sensor_pos.shape[0],
        labels=np.arange(sensor_pos.shape[0]).astype(str),
    ),
    static=True,
    strict=True,
)
rr.log(
    "traj/line",
    rr.LineStrips3D(strips=target_pos, radii=[0.05], colors=[0x43AAFFC3]),
    static=True,
    strict=True,
)
rr.log(
    "traj/ukf",
    rr.Points3D(
        positions=state_ukf.mean[:3],
        radii=[0.2],
        colors=[(0, 255, 0)],
    ),
)
rr.log("traj/cloud_fpf", rr.Points3D(positions=state_fpf.particles[:, :3]))


for i in range(seq_len):
    key, sub_key = jr.split(key)
    true_pos = target_pos[i, :]
    rr.log(
        "traj/true",
        rr.Points3D(positions=true_pos, radii=[0.1], colors=[(255, 0, 0)]),
    )

    meas_noisy = meas_fn(true_pos) + jr.uniform(
        key=sub_key, shape=(meas_dim,), minval=-1, maxval=1
    ) * jnp.sqrt(jnp.diag(R))

    log_measurement(sensor_type, meas_noisy, sensor_pos)

    state_pred_ukf = filter_ukf.predict(state_ukf)
    state_ukf = filter_ukf.update(state_pred_ukf, meas_noisy)
    ukf_hist.append(state_ukf)

    state_fpf, flow_states = filter_fpf.run_step(state_fpf, meas_noisy)
    fpf_hist.append(state_fpf)

    rr.log("traj/ukf", rr.Points3D(positions=state_ukf.mean[:3]))
    rr.log("traj/cloud_fpf", rr.Points3D(positions=state_fpf.particles[:, :3]))
    p_flow = flow_states[0][..., :3].transpose(1,0,2)
    rr.log("traj/homotopy_flow", rr.LineStrips3D(strips=np.array(p_flow), radii=[0.01]*n_particles),)

rr.notebook_show(width=960, height=720)

HTML(value='<div id="4ec45cb5-fb9f-483d-b54d-83dbbdd3ed8a"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…